In [5]:
import pandas as pd
import numpy as np

# ----------------------------
# 1) Income binning
# ----------------------------
def map_income(hincp):
    if pd.isna(hincp):
        return np.nan
    x = float(hincp)
    if x < 0:
        return np.nan
    elif x < 15000:
        return "<15k"
    elif x < 25000:
        return "15k-25k"
    elif x < 35000:
        return "25k-35k"
    elif x < 50000:
        return "35k-50k"
    elif x < 100000:
        return "50k-100k"
    elif x < 200000:
        return "100k-200k"
    else:
        return "200k+"

# ----------------------------
# 2) Build income lookup from 2020 HUS
# ----------------------------
house_files_2020 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husa.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husb.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husc.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husd.csv",
]

def build_income_lookup(file_paths, chunksize=100000):
    parts = []

    for fp in file_paths:
        for chunk in pd.read_csv(
            fp,
            usecols=["SERIALNO", "HINCP"],
            chunksize=chunksize,
            low_memory=False,
            dtype={"SERIALNO": "string"}
        ):
            chunk["HINCP"] = pd.to_numeric(chunk["HINCP"], errors="coerce")
            chunk["income"] = chunk["HINCP"].apply(map_income)
            chunk = chunk.dropna(subset=["SERIALNO", "income"])
            parts.append(chunk[["SERIALNO", "income"]])

    income_lookup = pd.concat(parts, ignore_index=True)
    income_lookup = income_lookup.drop_duplicates(subset=["SERIALNO"])
    return income_lookup

income_lookup_2020 = build_income_lookup(house_files_2020)
income_lookup_2020["SERIALNO"] = income_lookup_2020["SERIALNO"].astype("string")
income_lookup_2020.to_csv("income_lookup_2020.csv", index=False)

print(income_lookup_2020.head())
print(income_lookup_2020["income"].value_counts().sort_index())

# ----------------------------
# 3) Your existing person-side mappings
# ----------------------------
GROUP_COLS = ["age", "sex", "race", "education", "income"]
USECOLS = ["PWGTP", "AGEP", "SEX", "HISP", "RAC1P", "SCHL", "SERIALNO"]

def map_age(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if 18 <= x <= 24: return "18-24"
    elif 25 <= x <= 29: return "25-29"
    elif 30 <= x <= 34: return "30-34"
    elif 35 <= x <= 39: return "35-39"
    elif 40 <= x <= 44: return "40-44"
    elif 45 <= x <= 49: return "45-49"
    elif 50 <= x <= 54: return "50-54"
    elif 55 <= x <= 59: return "55-59"
    elif 60 <= x <= 64: return "60-64"
    elif 65 <= x <= 69: return "65-69"
    elif 70 <= x <= 74: return "70-74"
    elif 75 <= x <= 79: return "75-79"
    elif x >= 80: return "80+"
    return np.nan

def map_sex(x):
    return {1: "Male", 2: "Female"}.get(x, np.nan)

def map_edu(x):
    if pd.isna(x):
        return np.nan
    x = int(x)
    if x <= 15:
        return "Did not graduate high school"
    elif x <= 17:
        return "Graduated high school"
    elif x <= 20:
        return "Attended college or technical school"
    else:
        return "Graduated college or technical school"

def map_race(row):
    hisp = row["HISP"]
    rac = row["RAC1P"]

    if pd.isna(hisp) or pd.isna(rac):
        return np.nan

    # Hispanic overrides race
    if int(hisp) != 1:
        return "Hispanic"

    rac = int(rac)

    if rac == 1:
        return "NH-White"
    elif rac == 2:
        return "NH-Black"
    elif rac == 3:
        return "AIAN"
    elif rac in [4, 5, 6, 7, 8, 9, 10]:
        return "Asian"
    elif rac in [11, 12, 13, 14]:
        return "NHOPI"
    elif rac == 15:
        return "Other"
    else:
        return "Multiracial"

# ----------------------------
# 4) Process one person chunk, merge binned income lookup, then group
# ----------------------------
def process_person_chunk(chunk, income_lookup):
    chunk = chunk.copy()
    
    chunk["SERIALNO"] = chunk["SERIALNO"].astype("string")
    income_lookup["SERIALNO"] = income_lookup["SERIALNO"].astype("string")

    chunk["weight"] = pd.to_numeric(chunk["PWGTP"], errors="coerce")
    chunk["age"] = chunk["AGEP"].apply(map_age)
    chunk["sex"] = chunk["SEX"].apply(map_sex)
    chunk["race"] = chunk.apply(map_race, axis=1)
    chunk["education"] = chunk["SCHL"].apply(map_edu)

    chunk = chunk.merge(income_lookup, on="SERIALNO", how="left")
    chunk = chunk.dropna(subset=["age", "sex", "race", "education", "income", "weight"])

    grouped = (
        chunk.groupby(GROUP_COLS, observed=True)["weight"]
        .sum()
        .reset_index()
    )
    return grouped

# ----------------------------
# 5) Build final 2020 joint table from person files
# ----------------------------
person_files_2020 = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd.csv",
]

partials = []

for fp in person_files_2020:
    for chunk in pd.read_csv(
        fp,
        usecols=USECOLS,
        chunksize=100000,
        low_memory=False,
        dtype={"SERIALNO": "string"}
    ):
        partials.append(process_person_chunk(chunk, income_lookup_2020))

joint_2020 = pd.concat(partials, ignore_index=True)
joint_2020 = joint_2020.groupby(GROUP_COLS, as_index=False)["weight"].sum()
joint_2020["year"] = 2020
joint_2020["prop"] = joint_2020["weight"] / joint_2020["weight"].sum()

joint_2020 = joint_2020.sort_values(GROUP_COLS).reset_index(drop=True)

joint_2020.to_csv("pums_joint_2020_complete.csv", index=False)

print(joint_2020.head())
print(joint_2020["prop"].sum())
print(joint_2020.shape)

        SERIALNO    income
0  2016000000064   35k-50k
1  2016000000080   35k-50k
2  2016000000107  50k-100k
3  2016000000134  50k-100k
4  2016000000180  50k-100k
income
100k-200k    1318301
15k-25k       519741
200k+         504392
25k-35k       518935
35k-50k       738200
50k-100k     1823588
<15k          592068
Name: count, dtype: int64
     age     sex  race                             education     income  \
0  18-24  Female  AIAN  Attended college or technical school  100k-200k   
1  18-24  Female  AIAN  Attended college or technical school    15k-25k   
2  18-24  Female  AIAN  Attended college or technical school      200k+   
3  18-24  Female  AIAN  Attended college or technical school    25k-35k   
4  18-24  Female  AIAN  Attended college or technical school    35k-50k   

   weight  year      prop  
0    6579  2020  0.000027  
1    3994  2020  0.000016  
2     855  2020  0.000003  
3    3101  2020  0.000013  
4    4917  2020  0.000020  
0.9999999999999999
(3640, 8)


In [7]:
def process_year(person_files, house_files, year, out_file):
    income_lookup = build_income_lookup(house_files)
    income_lookup["SERIALNO"] = income_lookup["SERIALNO"].astype("string")

    partials = []

    for fp in person_files:
        for chunk in pd.read_csv(
            fp,
            usecols=USECOLS,
            chunksize=100000,
            low_memory=False,
            dtype={"SERIALNO": "string"}
        ):
            processed = process_person_chunk(chunk, income_lookup)
            partials.append(processed)

    joint = pd.concat(partials, ignore_index=True)
    joint = joint.groupby(GROUP_COLS, as_index=False)["weight"].sum()
    joint["year"] = year
    joint["prop"] = joint["weight"] / joint["weight"].sum()

    joint = joint.sort_values(GROUP_COLS).reset_index(drop=True)
    joint.to_csv(out_file, index=False)

    return joint

In [9]:
files_2021_person = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2021.csv"
]

files_2021_house = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husa_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husb_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husc_2021.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husd_2021.csv",
]

files_2022_person = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2022.csv"
]

files_2022_house = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husa_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husb_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husc_2022.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husd_2022.csv",
]

files_2023_person = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2023.csv"
]

files_2023_house = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husa_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husb_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husc_2023.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husd_2023.csv",
]

files_2024_person = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusa_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusb_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusc_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_pusd_2024.csv"
]

files_2024_house = [
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husa_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husb_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husc_2024.csv",
    r"C:\Users\kband\Documents\ACS2020-2021\psam_husd_2024.csv",
]

joint_2021 = process_year(files_2021_person, files_2021_house, 2021, "pums_joint_2021.csv")
joint_2022 = process_year(files_2022_person, files_2022_house, 2022, "pums_joint_2022.csv")
joint_2023 = process_year(files_2023_person, files_2023_house, 2023, "pums_joint_2023.csv")
joint_2024 = process_year(files_2024_person, files_2024_house, 2024, "pums_joint_2024.csv")

In [11]:
print(joint_2021["prop"].sum())
print(joint_2022["prop"].sum())
print(joint_2023["prop"].sum())
print(joint_2024["prop"].sum())

1.0
1.0
1.0
1.0


In [13]:
joint_all = pd.concat(
    [joint_2020, joint_2021, joint_2022, joint_2023, joint_2024],
    ignore_index=True
)


In [15]:
joint_all.to_csv("pums_joint_2020_2024.csv", index=False)

In [17]:
joint_all.groupby("year")["prop"].sum()

year
2020    1.0
2021    1.0
2022    1.0
2023    1.0
2024    1.0
Name: prop, dtype: float64